# Graph-CoT: Neo4j 최적화와 vLLM 재진입

GPU·DB·API key 없이 기존 결과와 tool 계약을 읽는 튜토리얼입니다. **새 성능 실험이 아닙니다.**

[전체 실습](../examples/graphcot_optimization/TUTORIAL.md) · [원리/결과](../examples/graphcot_optimization/RESULTS.md)

저장소를 clone한 뒤 repository root 또는 tutorials/에서 실행하세요. 첫 두 실습은 표준 라이브러리만 필요합니다.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'examples/graphcot_optimization/lab.py').exists())
LAB = ROOT / 'examples/graphcot_optimization'
result = subprocess.run([sys.executable, str(LAB / 'lab.py'), 'verify'],
                        capture_output=True, text=True, check=True)
print(result.stdout)


## 1. Tool call → checked Cypher → oracle

합성 graph의 병렬 송금2개가 DISTINCT 수취인1행으로 나오는 계약입니다. 실제 Neo4j 실행이 아닙니다.


In [ ]:
result = subprocess.run([sys.executable, str(LAB / 'lab.py'), 'demo'],
                        capture_output=True, text=True, check=True)
demo = json.loads(result.stdout)
print(demo['bound']['query'])
print(demo['bound']['parameters'])
print('Oracle:', demo['oracle_rows'])
print('Live performance evidence:', demo['performance_evidence'])


## 2. DB 개선이 E2E 개선으로 이어졌나요?

아래 값은 저장된 실제 BF16/neo4rs 연구의 요약입니다. Notebook이 inference를 재실행하는 것은 아닙니다. 동일 모델과 memory 두 조건, 3block, finite cohort입니다.


In [ ]:
evidence = json.loads((LAB / 'evidence/memory-e2e.json').read_text())
for c in evidence['comparisons']:
    print(f"{c['workload']:8s} C{c['concurrency']:2d}: "
          f"{c['baseline_batch_seconds']:.3f} → {c['candidate_batch_seconds']:.3f}s "
          f"({c['candidate_batch_change_percent']:+.2f}%)")
print('Measured correct/offered:', evidence['correct'], '/', evidence['measured'])


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(LAB / 'assets/neo4j-to-vllm-overview.png')))


## 3. 다음 연구를 설계하세요

- 사용자C와 실제 tool/DB 동시성을 구분하세요.
- 도구 반환 후 enqueue/first schedule/completion의 절대시점을 비교하세요.
- offered=ok+failed+dropped, warmup 별도, evidence/token 동등성을 확인하세요.
- Batch drain과 p95/공정성을 함께 보세요.
- [live 실습 조건](../examples/graphcot_optimization/TUTORIAL.md#5-실제-db-실험--소유한-환경에서만-선택-실행)과 [vLLM 인수인계](../examples/graphcot_optimization/VLLM_HANDOFF.md)를 확인하세요.

Offline 반례/demo 성공은 DB 호환성·처리율·production readiness의 증거가 아닙니다.
